# 04 - Training Pipeline
## PhishScamSense: Real-Time Multimodal Phishing Defense

This notebook covers:
1. Loading a training subset from the CIC-Bell-DNS2021 dataset
2. Preparing tokenized + numerical features for all 4 classes
3. Pre-training the fusion model (neural network branches) with CrossEntropyLoss
4. Training the XGBoost 4-class classifier on fused embeddings
5. MLflow experiment tracking and model checkpointing

> **Full training:** run `python -m ml.src.training.train --mode xgboost` from the project root.
> This notebook uses a small subset (≤2,000 per class) for demonstration.

In [ ]:
import sys
import os
import logging
from pathlib import Path

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
sys.path.insert(0, PROJECT_ROOT)

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import xgboost as xgb
import matplotlib.pyplot as plt

from ml.src.data.data_loader import load_cic_bell_dns2021
from ml.src.features.url_features import extract_url_features

logging.basicConfig(level=logging.INFO, format="%(asctime)s  %(levelname)-8s  %(message)s", datefmt="%H:%M:%S")
logger = logging.getLogger(__name__)

CLASS_NAMES  = ["benign", "phishing", "malware", "spam"]
NUM_CLASSES  = 4
DATA_DIR     = Path(PROJECT_ROOT) / "data" / "raw"
CHECKPOINT_DIR = Path(PROJECT_ROOT) / "ml" / "checkpoints"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

## 4.1 Define All Model Components

Re-define model classes here so this notebook is self-contained and runnable.

In [ ]:
from transformers import DistilBertModel, DistilBertTokenizer

# ── Model classes (mirrors ml/src/models/) ──────────────────────────────────

class AttentionLayer(nn.Module):
    def __init__(self, hidden_size):
        super().__init__()
        self.attention = nn.Linear(hidden_size, 1)
    def forward(self, lstm_output):
        weights = torch.softmax(self.attention(lstm_output), dim=1)
        return torch.sum(weights * lstm_output, dim=1)

class NLPBranch(nn.Module):
    def __init__(self, output_dim=128, lstm_hidden=256, lstm_layers=2, dropout=0.3, freeze_bert=True):
        super().__init__()
        self.distilbert = DistilBertModel.from_pretrained("distilbert-base-uncased")
        if freeze_bert:
            for param in self.distilbert.parameters():
                param.requires_grad = False
        bert_hidden = self.distilbert.config.hidden_size
        self.bilstm = nn.LSTM(bert_hidden, lstm_hidden, lstm_layers, batch_first=True,
                              bidirectional=True, dropout=dropout if lstm_layers > 1 else 0)
        self.attention = AttentionLayer(lstm_hidden * 2)
        self.fc = nn.Linear(lstm_hidden * 2, output_dim)
        self.dropout = nn.Dropout(dropout)
    def forward(self, input_ids, attention_mask):
        bert_out = self.distilbert(input_ids=input_ids, attention_mask=attention_mask)
        lstm_out, _ = self.bilstm(bert_out.last_hidden_state)
        attended = self.attention(lstm_out)
        return self.fc(self.dropout(attended))

class MLPBranch(nn.Module):
    def __init__(self, input_dim=23, hidden_dims=None, output_dim=64, dropout=0.3):
        super().__init__()
        if hidden_dims is None:
            hidden_dims = [128, 64]
        layers = []
        prev_dim = input_dim
        for h in hidden_dims:
            layers.extend([nn.Linear(prev_dim, h), nn.BatchNorm1d(h), nn.ReLU(), nn.Dropout(dropout)])
            prev_dim = h
        layers.append(nn.Linear(prev_dim, output_dim))
        self.network = nn.Sequential(*layers)
    def forward(self, x):
        return self.network(x)

class PhishScamSenseFusionModel(nn.Module):
    def __init__(self, num_features=23, nlp_output_dim=128, numerical_output_dim=64, freeze_bert=True):
        super().__init__()
        self.nlp_branch = NLPBranch(output_dim=nlp_output_dim, freeze_bert=freeze_bert)
        self.numerical_branch = MLPBranch(input_dim=num_features, output_dim=numerical_output_dim)
        self.fusion_dim = nlp_output_dim + numerical_output_dim
    def forward(self, input_ids, attention_mask, numerical_features):
        nlp_out = self.nlp_branch(input_ids, attention_mask)
        num_out = self.numerical_branch(numerical_features)
        return torch.cat([nlp_out, num_out], dim=1)

class URLTokenizer:
    def __init__(self, max_length=128):
        self.tokenizer = DistilBertTokenizer.from_pretrained("distilbert-base-uncased")
        self.max_length = max_length
    def tokenize(self, urls):
        return self.tokenizer(urls, padding=True, truncation=True,
                              max_length=self.max_length, return_tensors="pt")

print("All model components defined.")

## 4.2 Load CIC-Bell-DNS2021 Training Subset

We cap each class to keep this notebook runnable in reasonable time.
For full training use `ml.src.training.train`.

In [ ]:
import random
from sklearn.model_selection import train_test_split

SAMPLES_PER_CLASS = 2_000   # increase for better accuracy; decrease for faster runs
SEED = 42

# Load full dataset (benign capped) then subsample per class for the notebook
urls_all, labels_all = load_cic_bell_dns2021(DATA_DIR, max_benign=SAMPLES_PER_CLASS * 2, seed=SEED)

rng = random.Random(SEED)
per_class: dict[int, list[str]] = {i: [] for i in range(NUM_CLASSES)}
for url, lbl in zip(urls_all, labels_all):
    per_class[lbl].append(url)

urls, labels = [], []
for lbl, url_list in per_class.items():
    sample = rng.sample(url_list, min(SAMPLES_PER_CLASS, len(url_list)))
    urls.extend(sample)
    labels.extend([lbl] * len(sample))
    logger.info(f"  {CLASS_NAMES[lbl]:12s}: {len(sample):,}")

urls   = np.array(urls,   dtype=object)
labels = np.array(labels, dtype=np.int32)
logger.info(f"Notebook training set: {len(urls):,} URLs")

# Train / val split
X_train, X_val, y_train, y_val = train_test_split(
    urls, labels, test_size=0.15, random_state=SEED, stratify=labels
)
logger.info(f"Train: {len(X_train):,}   Val: {len(X_val):,}")

# Prepare dataset helper
def prepare_dataset(url_arr, label_arr):
    tokenizer = URLTokenizer()
    tokens = tokenizer.tokenize(url_arr.tolist())
    num_feat = [list(extract_url_features(u).values()) for u in url_arr]
    return {
        "input_ids":          tokens["input_ids"],
        "attention_mask":     tokens["attention_mask"],
        "numerical_features": torch.tensor(num_feat, dtype=torch.float32),
        "labels":             label_arr,
    }

logger.info("Tokenizing & extracting features for training set…")
train_data = prepare_dataset(X_train, y_train)
logger.info("Tokenizing & extracting features for validation set…")
val_data   = prepare_dataset(X_val, y_val)

print(f"\ninput_ids shape:          {train_data['input_ids'].shape}")
print(f"numerical_features shape: {train_data['numerical_features'].shape}")
print(f"labels unique:            {np.unique(train_data['labels'])}")

## 4.3 Train Fusion Model (Neural Network Branches)

Pre-train the neural branches using **CrossEntropyLoss** over all 4 classes.
The learned embeddings are later used as input to the XGBoost classifier.

In [ ]:
def train_fusion_model(fusion_model, train_data, val_data,
                       epochs=5, batch_size=16, learning_rate=1e-4, device="cpu"):
    """Pre-train the fusion model branches with 4-class CrossEntropyLoss."""
    fusion_model = fusion_model.to(device)
    classifier_head = nn.Linear(fusion_model.fusion_dim, NUM_CLASSES).to(device)

    optimizer = torch.optim.Adam(
        list(fusion_model.parameters()) + list(classifier_head.parameters()),
        lr=learning_rate,
    )
    criterion = nn.CrossEntropyLoss()

    dataset = TensorDataset(
        train_data["input_ids"],
        train_data["attention_mask"],
        train_data["numerical_features"],
        torch.tensor(train_data["labels"], dtype=torch.long),
    )
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

    # Validation tensors
    val_ids  = val_data["input_ids"].to(device)
    val_mask = val_data["attention_mask"].to(device)
    val_num  = val_data["numerical_features"].to(device)
    val_lbl  = torch.tensor(val_data["labels"], dtype=torch.long).to(device)

    history = {"train_loss": [], "val_loss": [], "val_acc": []}

    for epoch in range(epochs):
        fusion_model.train(); classifier_head.train()
        total_loss = 0.0
        for ids, mask, num_feat, lbls in dataloader:
            ids, mask, num_feat, lbls = ids.to(device), mask.to(device), num_feat.to(device), lbls.to(device)
            optimizer.zero_grad()
            logits = classifier_head(fusion_model(ids, mask, num_feat))
            loss   = criterion(logits, lbls)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        avg_train_loss = total_loss / len(dataloader)

        # Validation
        fusion_model.eval(); classifier_head.eval()
        with torch.no_grad():
            val_logits = classifier_head(fusion_model(val_ids, val_mask, val_num))
            val_loss   = criterion(val_logits, val_lbl).item()
            val_acc    = (val_logits.argmax(dim=1) == val_lbl).float().mean().item()

        history["train_loss"].append(avg_train_loss)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)
        logger.info(f"Epoch {epoch+1}/{epochs}  train_loss={avg_train_loss:.4f}  "
                    f"val_loss={val_loss:.4f}  val_acc={val_acc:.4f}")

    return fusion_model, history

print("Training function defined.")

## 4.4 Run Training with MLflow Tracking

In [ ]:
try:
    import mlflow
    MLFLOW_TRACKING_URI = os.path.join(PROJECT_ROOT, "ml", "mlruns")
    mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
    mlflow.set_experiment("phishscamsense")
    USE_MLFLOW = True
except Exception:
    USE_MLFLOW = False
    logger.warning("MLflow not available — skipping experiment tracking.")

EPOCHS     = 5
BATCH_SIZE = 16
LR         = 1e-4

run_ctx = mlflow.start_run(run_name="fusion_model_v1") if USE_MLFLOW else __import__("contextlib").nullcontext()

with run_ctx:
    if USE_MLFLOW:
        mlflow.log_params({
            "epochs": EPOCHS, "batch_size": BATCH_SIZE, "learning_rate": LR,
            "model_type": "DistilBERT+BiLSTM+Attention+MLP+XGBoost",
            "num_classes": NUM_CLASSES, "classes": str(CLASS_NAMES),
            "num_train": len(X_train), "num_val": len(X_val),
        })

    # Step 1: Train fusion model
    logger.info("Step 1: Training fusion model (neural network branches)…")
    fusion_model = PhishScamSenseFusionModel(num_features=23, freeze_bert=True)
    fusion_model, history = train_fusion_model(
        fusion_model, train_data, val_data,
        epochs=EPOCHS, batch_size=BATCH_SIZE, learning_rate=LR, device=device,
    )
    if USE_MLFLOW:
        for step, (tl, vl, va) in enumerate(zip(history["train_loss"], history["val_loss"], history["val_acc"])):
            mlflow.log_metrics({"train_loss": tl, "val_loss": vl, "val_acc": va}, step=step)

    # Step 2: Extract fused embeddings for XGBoost
    logger.info("Step 2: Extracting fused embeddings…")
    fusion_model.eval()
    def embed(data):
        with torch.no_grad():
            return fusion_model(
                data["input_ids"].to(device),
                data["attention_mask"].to(device),
                data["numerical_features"].to(device),
            ).cpu().numpy()

    train_emb = embed(train_data)
    val_emb   = embed(val_data)
    logger.info(f"Fused embedding shape: {train_emb.shape}")

    # Step 3: Train XGBoost 4-class
    logger.info("Step 3: Training XGBoost 4-class classifier…")
    xgb_clf = xgb.XGBClassifier(
        n_estimators=200, max_depth=6, learning_rate=0.1,
        objective="multi:softmax", num_class=NUM_CLASSES,
        eval_metric="mlogloss", early_stopping_rounds=10, n_jobs=-1,
    )
    xgb_clf.fit(train_emb, train_data["labels"],
                eval_set=[(val_emb, val_data["labels"])], verbose=50)

    # Step 4: Training accuracy
    from sklearn.metrics import accuracy_score
    train_acc = accuracy_score(train_data["labels"], xgb_clf.predict(train_emb))
    val_acc   = accuracy_score(val_data["labels"],   xgb_clf.predict(val_emb))
    if USE_MLFLOW:
        mlflow.log_metrics({"xgb_train_acc": train_acc, "xgb_val_acc": val_acc})

    logger.info(f"XGBoost train accuracy: {train_acc:.4f}")
    logger.info(f"XGBoost val  accuracy:  {val_acc:.4f}")

print("\nTraining complete!")

## 4.5 Plot Training Loss & Save Checkpoints

In [ ]:
import pickle

# Plot training curves
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

epochs_range = range(1, EPOCHS + 1)
axes[0].plot(epochs_range, history["train_loss"], marker="o", label="Train", color="#3498db")
axes[0].plot(epochs_range, history["val_loss"],   marker="s", label="Val",   color="#e74c3c")
axes[0].set_title("Fusion Model Loss (CrossEntropyLoss — 4 classes)")
axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Loss")
axes[0].legend(); axes[0].grid(True)

axes[1].plot(epochs_range, history["val_acc"], marker="o", color="#2ecc71")
axes[1].set_title("Validation Accuracy (4-class)")
axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Accuracy")
axes[1].set_ylim(0, 1.05); axes[1].grid(True)

plt.tight_layout()
plt.show()

# Save checkpoints
torch.save(fusion_model.state_dict(), CHECKPOINT_DIR / "fusion_model.pt")
with open(CHECKPOINT_DIR / "xgb_classifier.pkl", "wb") as f:
    pickle.dump(xgb_clf, f)
xgb_clf.save_model(str(CHECKPOINT_DIR / "xgb_classifier.json"))

print(f"Checkpoints saved to: {CHECKPOINT_DIR}")
print("  fusion_model.pt")
print("  xgb_classifier.pkl")
print("  xgb_classifier.json")